# Presentation figures

Slide-ready figures for the main results. Reads the saved model outputs and fits nothing:

- `mixed_model.py` outputs (`fit_mixed_model.slurm`): the before/after and BACI effect per measure
- the primary distance run (`fit_distance_models.slurm`, `stage3/primary__auto/`): how effects change downstream

Each figure is shown here and saved as a PNG (200 dpi) in `data/processed/presentation_figures/<case_study>/`.
Detailed diagnostics stay in `mixed_model_results.ipynb` and `distance_model_results.ipynb`.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "etl").is_dir() and (p / "config").is_dir())
sys.path.insert(0, str(ROOT))
from analysis import mixed_model as mm
from analysis import stage4_report as rep

CASE_STUDY = "all_removals"
MODEL_DIR = Path(os.environ.get("DAM_REMOVAL_MODEL_DIR", mm.output_dir_for(CASE_STUDY, ROOT)))
FIG_DIR = Path(os.environ.get("DAM_REMOVAL_FIG_DIR",
                           ROOT / "data" / "processed" / "presentation_figures" / CASE_STUDY))
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Label for the figure footers: which fit these results come from.
RUN_LABEL = "Final model fit, 2026-11-17 (corrected dam locations and station reaches)"

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
    "font.size": 12, "axes.titlesize": 13, "axes.titleweight": "bold", "axes.labelsize": 12,
    "xtick.labelsize": 11, "ytick.labelsize": 11, "legend.fontsize": 11,
    "axes.spines.top": False, "axes.spines.right": False,
})

NAMES = {
    "water_temp": "Water temperature",
    "dissolved_oxygen": "Dissolved oxygen",
    "ph": "pH",
    "specific_conductance": "Specific conductance",
    "turbidity": "Turbidity",
    "phosphorus (Total)": "Total phosphorus",
    "orthophosphate (Dissolved)": "Dissolved orthophosphate",
    "tds": "Total dissolved solids",
    "ssc": "Suspended sediment (SSC)",
    "tss": "Total suspended solids",
}
# The measures shown in the headline figures, in this order.
CORE = ["water_temp", "dissolved_oxygen", "ph", "specific_conductance",
        "turbidity", "phosphorus (Total)", "orthophosphate (Dissolved)"]

DESIGNS = {
    "before_after": ("Downstream, before vs. after", "#1f6f8b"),
    "baci": ("Downstream vs. upstream controls (BACI)", "#d4762c"),
}
MIN_DAMS_SOLID = 15     # below this, an estimate is labelled exploratory


def save(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png")
    print(f"Saved {FIG_DIR / name}.png")


def footer(fig, text, y=-0.02, width=None):
    import textwrap
    width = width or int(fig.get_figwidth() * 15)
    lines = [w for part in text.split("\n") for w in textwrap.wrap(part, width)]
    fig.text(0.01, y, "\n".join(lines), ha="left", va="top", fontsize=9.5, color="0.35")


def to_display(values, log_scale):
    """Model units -> slide units: % change for log10-scale measures."""
    values = np.asarray(values, dtype=float)
    return (10 ** values - 1) * 100 if log_scale else values


def unit_of(series, log_scale, unit=""):
    return "% change" if log_scale else {"deg C": "°C", "std units": "pH units"}.get(unit, unit)


def threshold_text(series):
    return rep.sesoi_label(series).replace("+/-", "±").replace("deg C", "°C").replace("std units", "pH units")

## 1. Headline: what changed after removal

One panel per measure, in its own units. **Dot** = average change across dams after removal;
**bar** = 95% confidence interval. **Filled** dots = the interval excludes zero (a detectable change);
**hollow** = can't distinguish from no change. Numbers in brackets = dams.

In [ ]:
results = pd.read_csv(MODEL_DIR / "results.csv")
results = results[results["error"].fillna("").eq("") & results["estimate"].notna()].copy()
log = results["log_scale"].astype(str).str.lower().eq("true")
results["est"] = np.where(log, results["pct_estimate"], results["estimate"])
results["lo"] = np.where(log, results["pct_ci_low"], results["ci_low"])
results["hi"] = np.where(log, results["pct_ci_high"], results["ci_high"])
results["log"] = log
results["detected"] = (results["lo"] > 0) | (results["hi"] < 0)


def headline(frame, series_list, title, name, ncols=4):
    series_list = [s for s in series_list if s in set(frame["series"])]
    nrows = int(np.ceil(len(series_list) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.1 * ncols, 2.5 * nrows), squeeze=False)
    for ax, series in zip(axes.flat, series_list):
        rows = frame[frame["series"] == series].set_index("model").reindex(list(DESIGNS)).dropna(subset=["est"])
        for i, (model, r) in enumerate(rows.iloc[::-1].iterrows()):
            label, color = DESIGNS[model]
            ax.plot([r["lo"], r["hi"]], [i, i], color=color, lw=2.5, solid_capstyle="round")
            ax.plot(r["est"], i, "o", ms=10, mec=color, mew=2,
                    mfc=color if r["detected"] else "white", zorder=3)
            tag = f"[{int(r['dams'])}]" + ("*" if r["dams"] < MIN_DAMS_SOLID else "")
            ax.annotate(tag, (r["hi"], i), xytext=(6, 0), textcoords="offset points",
                        va="center", fontsize=10, color="0.3")
        ax.axvline(0, color="0.2", lw=1)
        ax.set_yticks([])
        ax.set_ylim(-0.7, len(rows) - 0.3)
        r0 = rows.iloc[0]
        ax.set_title(NAMES.get(series, series), loc="left")
        ax.set_xlabel(unit_of(series, r0["log"], r0["unit"]))
        lo, hi = rows["lo"].min(), rows["hi"].max()
        pad = 0.25 * (hi - lo)
        ax.set_xlim(min(lo, 0) - pad * 0.3, max(hi, 0) + pad)
    for ax in list(axes.flat)[len(series_list):]:
        ax.set_visible(False)
    handles = [Line2D([], [], color=c, marker="o", ms=9, lw=2.5, label=l) for l, c in DESIGNS.values()]
    handles += [Line2D([], [], color="0.3", marker="o", ms=9, lw=0, label="Detectable change (filled)"),
                Line2D([], [], color="0.3", marker="o", ms=9, mfc="white", lw=0,
                       label="No detectable change (hollow)")]
    fig.legend(handles=handles, loc="upper center", ncol=2, frameon=False,
               bbox_to_anchor=(0.5, 1.0 + 0.09 / nrows))
    fig.suptitle(title, fontsize=16, fontweight="bold", y=1.0 + 0.2 / nrows)
    fig.tight_layout()
    footer(fig, "Average change in the 5 years after removal vs. the 5 years before, with 95% confidence "
                "intervals. Mixed-effects models (dam and station random effects; month, air temperature, rain).\n"
                f"[n] = dams; * = fewer than {MIN_DAMS_SOLID} dams (exploratory).   {RUN_LABEL}")
    save(fig, name)
    plt.show()


headline(results, CORE, "How water quality changed after dam removal", "01_headline_effects")

## 2. The two clearest results, in plain numbers

Before/after downstream, the design with the most dams: water was cooler and held more oxygen after removal.

In [ ]:
KEY = ["water_temp", "dissolved_oxygen"]
ba = results[(results["model"] == "before_after") & results["series"].isin(KEY)].set_index("series")

fig, axes = plt.subplots(1, len(KEY), figsize=(10, 3.4))
for ax, series in zip(np.atleast_1d(axes), KEY):
    if series not in ba.index:
        ax.set_visible(False); continue
    r = ba.loc[series]
    unit = unit_of(series, r["log"], r["unit"])
    color = "#2b7bba" if r["est"] < 0 else "#2a9d55"
    if not r["detected"]:
        color = "0.65"
    ax.barh([0], [r["est"]], height=0.5, color=color, alpha=0.85)
    ax.errorbar(r["est"], 0, xerr=[[r["est"] - r["lo"]], [r["hi"] - r["est"]]], fmt="none",
                ecolor="0.15", capsize=8, lw=2)
    ax.axvline(0, color="0.2", lw=1)
    ax.set_yticks([])
    ax.set_ylim(-0.6, 0.6)
    direction = ("warmer", "cooler") if series == "water_temp" else ("more oxygen", "less oxygen")
    word = (direction[0] if r["est"] > 0 else direction[1]) if r["detected"] else "no detectable change"
    ax.set_title(f"{NAMES[series]}: {r['est']:+.2f} {unit}\n", loc="left")
    ax.text(0, 1.02, f"{word}  ·  {int(r['dams'])} dams, {int(r['stations'])} stations",
            transform=ax.transAxes, ha="left", va="bottom", fontsize=11, color="0.3")
    span = max(abs(r["lo"]), abs(r["hi"])) * 1.3
    ax.set_xlim(-span, span)
    ax.set_xlabel(f"Change after removal ({unit})")
found = [n for n, s in [("cooler", "water_temp"), ("more oxygenated", "dissolved_oxygen")]
         if s in ba.index and ba.loc[s, "detected"]
         and (ba.loc[s, "est"] < 0 if s == "water_temp" else ba.loc[s, "est"] > 0)]
title = (f"Downstream water became {' and '.join(found)} after removal" if found
         else "Temperature and oxygen after removal")
fig.suptitle(title, fontsize=15, fontweight="bold", y=1.08)
fig.tight_layout()
footer(fig, "Before/after downstream model; bar = average change, whiskers = 95% confidence interval.   " + RUN_LABEL,
       y=-0.04)
save(fig, "02_temperature_oxygen")
plt.show()

## 3. Does every dam agree?

Each dam's own estimated change (the average plus that dam's deviation). Most dams moving the same way
supports a general removal effect; a wide spread means dams differ.

In [ ]:
dam_effects = pd.read_csv(MODEL_DIR / "dam_effects.csv")

fig, axes = plt.subplots(1, len(KEY), figsize=(11, 3.8))
for ax, series in zip(np.atleast_1d(axes), KEY):
    d = dam_effects[(dam_effects["series"] == series) & (dam_effects["model"] == "before_after")]["dam_effect"]
    if d.empty:
        ax.set_visible(False); continue
    unit = unit_of(series, False, ba.loc[series, "unit"]) if series in ba.index else ""
    lo, hi = d.quantile([0.02, 0.98])
    shown = d.clip(lo, hi)
    better = (d < 0).mean() if series == "water_temp" else (d > 0).mean()   # share in the average's direction
    ax.hist(shown, bins=25, color="#1f6f8b", alpha=0.85, edgecolor="white")
    ax.axvline(0, color="0.2", lw=1.2)
    ax.axvline(d.mean(), color="#d4762c", lw=2, ls="--", label=f"average {d.mean():+.2f} {unit}")
    direction = "cooler" if series == "water_temp" else "more oxygen"
    ax.set_title(f"{NAMES[series]}: {better:.0%} of {len(d)} dams {direction}", loc="left")
    ax.set_xlabel(f"Dam's estimated change after removal ({unit})")
    ax.set_ylabel("Number of dams")
    ax.legend(frameon=False, loc="upper right")
fig.suptitle("Most dams moved in the same direction, but individual dams vary", fontsize=15,
             fontweight="bold", y=1.04)
fig.tight_layout()
footer(fig, "Dam-specific effects from the before/after mixed model (shrunk toward the average for dams with "
            "little data); the most extreme 2% on each side are drawn at the edge.   " + RUN_LABEL, y=-0.04)
save(fig, "03_dam_agreement")
plt.show()

## 4. How far downstream do the effects reach?

From the primary distance model: the estimated change at each distance below the dam (line) with its 95%
confidence band. **Dots** = the estimate at 1, 5, 10, 20 and 30 km. **Dashed lines** = the provisional
practical threshold (changes inside them are too small to matter). Where the shaded band covers zero, we can't tell.

In [ ]:
SERIES_BY_DISTANCE = ["water_temp", "dissolved_oxygen", "ph"]   # edit to show other measures

stage3_results, stage3_effects, _ = rep.load_runs(MODEL_DIR / "stage3")
primary = stage3_results[(stage3_results["run"] == rep.PRIMARY[0]) & (stage3_results["re"] == rep.PRIMARY[1])]
curve = rep.effects_curve(primary)
points = rep.effects_by_distance(stage3_effects)
points = points[points["primary"] & points["distance_km"].notna()]
SPEC_TO_DESIGN = {"before/after": "before_after", "BACI": "baci"}


def distance_figure(series):
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.9), sharey=True)
    log_scale = series.split(" (")[0] in mm.SETTINGS["log_parameters"]
    unit = unit_of(series, log_scale, curve.loc[curve["series"] == series, "unit"].iloc[0]
                   if (curve["series"] == series).any() else "")
    sesoi = rep.SESOI.get(series)
    for ax, design in zip(axes, ["before/after", "BACI"]):
        label, color = DESIGNS[SPEC_TO_DESIGN[design]]
        c = curve[(curve["series"] == series) & (curve["design"] == design)]
        p = points[(points["series"] == series) & (points["design"] == design)]
        ax.axhline(0, color="0.2", lw=1)
        if sesoi is not None:
            band = to_display([-sesoi, sesoi], log_scale)
            for i, b in enumerate(band):
                ax.axhline(b, color="0.45", ls="--", lw=1.2,
                           label=f"Practical threshold ({threshold_text(series)})" if i == 0 else None)
        if c.empty:
            ax.text(0.5, 0.5, "Not estimated (too few dams)", transform=ax.transAxes, ha="center",
                    color="0.4")
        else:
            x = c["distance_km"]
            ax.fill_between(x, to_display(c["ci_low"], log_scale), to_display(c["ci_high"], log_scale),
                            color=color, alpha=0.2, lw=0, label="95% confidence band")
            ax.plot(x, to_display(c["estimate"], log_scale), color=color, lw=2.5, label="Estimated change")
            if len(p):
                est = to_display(p["estimate"], log_scale)
                ax.errorbar(p["distance_km"], est,
                            yerr=[est - to_display(p["ci_low"], log_scale),
                                  to_display(p["ci_high"], log_scale) - est],
                            fmt="o", color=color, capsize=4, ms=6)
            dams = int(c["dams"].iloc[0])
            label += f"  ({dams} dams{'; exploratory' if dams < MIN_DAMS_SOLID else ''})"
        ax.set_xscale("log")
        ax.set_xticks([1, 2, 5, 10, 20, 30], ["1", "2", "5", "10", "20", "30"])
        ax.set_xlabel("Distance downstream of the removed dam (km, log scale)")
        ax.set_title(label, loc="left", fontsize=12)
    axes[0].set_ylabel(f"Change after removal ({unit})")
    axes[0].legend(frameon=False, loc="best", fontsize=10)
    fig.suptitle(f"{NAMES.get(series, series)}: change after removal by distance downstream",
                 fontsize=15, fontweight="bold", y=1.05)
    fig.tight_layout()
    footer(fig, "Distance mixed-effects model (stations within 30 km; adjusted for dilution, season and "
                "weather). Threshold is a provisional analyst choice, not a regulatory standard.\n" + RUN_LABEL,
           y=-0.05)
    save(fig, f"04_distance_{rep.slug(series)}")
    plt.show()


for series in SERIES_BY_DISTANCE:
    distance_figure(series)

## 5. How much data each result rests on

Dams per measure in each design. BACI (with upstream controls) is the stronger design but has far fewer dams,
which is why several BACI intervals are wide.

In [ ]:
all_rows = pd.read_csv(MODEL_DIR / "results.csv")
counts = (all_rows[all_rows["series"].isin(NAMES)]
          .pivot_table(index="series", columns="model", values="dams", aggfunc="first")
          .reindex(columns=list(DESIGNS)).fillna(0))
counts = counts.loc[counts.sum(axis=1).sort_values().index]

fig, ax = plt.subplots(figsize=(10, 0.55 * len(counts) + 1.2))
y = np.arange(len(counts))
for offset, (model, (label, color)) in zip([0.2, -0.2], DESIGNS.items()):
    ax.barh(y + offset, counts[model], height=0.38, color=color, label=label)
    for yi, v in zip(y + offset, counts[model]):
        ax.text(v + 1.5, yi, f"{int(v)}", va="center", fontsize=10, color="0.25",
                bbox=dict(facecolor="white", edgecolor="none", pad=0.5))
ax.axvline(MIN_DAMS_SOLID, color="0.4", ls=":", lw=1.5, zorder=0)
ax.text(MIN_DAMS_SOLID, len(counts) - 0.4, f" {MIN_DAMS_SOLID} dams", color="0.4", fontsize=10, va="bottom")
ax.set_yticks(y, [NAMES[s] for s in counts.index])
ax.set_xlabel("Number of removed dams in the model")
ax.legend(frameon=False, loc="lower right")
ax.set_title("Sample size: dams per measure and design", loc="left", fontsize=15)
fig.tight_layout()
footer(fig, "Dams with enough monitoring before and after removal (≥3 samples in ≥2 years on each side), "
            "after model filters. Below 15 dams, results are exploratory.   " + RUN_LABEL, y=-0.03)
save(fig, "05_sample_size")
plt.show()

## 6. Robustness: did the location corrections change the answer?

The same models before the manual review (frozen 2026-11-10) and after correcting dam locations and station
reaches. Overlapping estimates = the conclusions don't depend on those corrections.

In [ ]:
FROZEN = Path(os.environ.get("DAM_REMOVAL_FROZEN_DIR",
                          ROOT / "data" / "processed" / "frozen" / "models-frozen-2026-11-10"))
frozen_path = next((p for p in sorted(FROZEN.rglob("results.csv")) if (p.parent / "dam_effects.csv").exists()),
                   None) if FROZEN.exists() else None

if frozen_path is None:
    print(f"No frozen mixed-model results under {FROZEN}; skipping.")
else:
    before = pd.read_csv(frozen_path)
    before = before[before["error"].fillna("").eq("") & before["estimate"].notna()].copy()
    blog = before["log_scale"].astype(str).str.lower().eq("true")
    before["est"] = np.where(blog, before["pct_estimate"], before["estimate"])
    before["lo"] = np.where(blog, before["pct_ci_low"], before["ci_low"])
    before["hi"] = np.where(blog, before["pct_ci_high"], before["ci_high"])
    both = results.merge(before[["series", "model", "est", "lo", "hi", "dams"]], on=["series", "model"],
                         suffixes=("", "_before"))
    shown = [s for s in CORE if s in set(both["series"])]
    fig, axes = plt.subplots(2, len(shown), figsize=(2.05 * len(shown), 6.2), squeeze=False)
    for row, model in enumerate(DESIGNS):
        label, color = DESIGNS[model]
        for col, series in enumerate(shown):
            ax = axes[row, col]
            r = both[(both["series"] == series) & (both["model"] == model)]
            if r.empty:
                ax.axis("off")
                ax.text(0.5, 0.5, "too few\ndams", ha="center", va="center", color="0.5",
                        transform=ax.transAxes, fontsize=10)
                continue
            r = r.iloc[0]
            for x, (e, l, h, filled) in enumerate([(r["est_before"], r["lo_before"], r["hi_before"], False),
                                                   (r["est"], r["lo"], r["hi"], True)]):
                ax.plot([x, x], [l, h], color=color, lw=2.2)
                ax.plot(x, e, "o", ms=8, mec=color, mew=2, mfc=color if filled else "white")
            ax.axhline(0, color="0.2", lw=0.8)
            ax.set_xlim(-0.6, 1.6)
            ax.set_xticks([0, 1], [f"before\n({int(r['dams_before'])})", f"after\n({int(r['dams'])})"],
                          fontsize=9)
            ax.tick_params(axis="y", labelsize=9)
            if row == 0:
                unit = unit_of(series, r["log"], r["unit"])
                ax.set_title(f"{NAMES[series]}\n({unit})", fontsize=10)
        axes[row, 0].annotate(label.replace(" (BACI)", "\n(BACI)").replace(", before", ",\nbefore"),
                              (-0.75, 0.5), xycoords="axes fraction", rotation=90, ha="center",
                              va="center", fontsize=10.5, fontweight="bold", color=color)
    fig.suptitle("Results before vs. after correcting dam and station locations", fontsize=15,
                 fontweight="bold", y=1.03)
    fig.tight_layout()
    footer(fig, "Hollow = frozen fit before the manual review (2026-11-10); filled = final fit. Bars = 95% "
                "confidence intervals; (n) = dams.\n181 dam locations corrected; all 723 analysis stations "
                "checked by hand, 24 reach assignments corrected.", y=-0.01)
    save(fig, "06_robustness_corrections")
    plt.show()

## Other figures worth reusing

- **Map of the dams** (interactive, red = BACI, blue = before/after only): `distance_model_results.ipynb`,
  "Where are the dams?". Take a screenshot zoomed to the lower 48.
- **Verdict grid by distance**: `distance_model_results.ipynb` section 2 (second figure).